# 37 Noise vs Nonlinearity

Notebook 31 made a toy distinction concrete:

`nearby states → amplification → threshold crossing → decision extraction`

This notebook asks the obvious next question:

> Is amplification just noise?

The goal is to compare deterministic state-dependent amplification against stochastic fluctuations.

This is still a toy model. It does not simulate semiclassical gravity, stochastic collapse, or the full Bao–Bouland–Jordan proof. It illustrates a computational distinction:

**reliable threshold crossing is different from random fluctuation.**

## 1. Setup paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"

RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

print(f"ROOT    = {ROOT}")
print(f"RESULTS = {RESULTS}")
print(f"FIGURES = {FIGURES}")

## 2. Load prior artifacts if available

In [ ]:
prior_paths = [
    RESULTS / "31_toy_discrimination.md",
    RESULTS / "23_bao_bouland_jordan.md",
    RESULTS / "29_quantum_gravity_implications.md",
]

for path in prior_paths:
    print(f"{'✓' if path.exists() else '✗'} {path.name}")

## 3. Model definitions

We compare three toy models for distinguishability \(d(t)\).

### Linear baseline

\[
d(t) = d_0
\]

### Deterministic nonlinear amplification

\[
d(t) = \frac{d_0 e^{kt}}{1 + d_0(e^{kt}-1)}
\]

This saturating form begins near \(d_0\), grows systematically, and stays bounded between 0 and 1.

### Stochastic noise comparison

The noisy trajectory fluctuates around \(d_0\) with a weak pullback term.

Noise can produce temporary excursions, but reliable decision extraction requires repeatable threshold crossing.

In [ ]:
def linear_baseline(t, d0=0.02):
    return np.full_like(t, d0, dtype=float)

def nonlinear_amplification(t, d0=0.02, k=0.65):
    return (d0 * np.exp(k * t)) / (1 + d0 * (np.exp(k * t) - 1))

def stochastic_noise(t, d0=0.02, sigma=0.006, pullback=0.06, seed=None):
    rng = np.random.default_rng(seed)
    noise = np.empty_like(t, dtype=float)
    noise[0] = d0

    for i in range(1, len(t)):
        step = rng.normal(0, sigma)
        restoring = -pullback * (noise[i - 1] - d0)
        noise[i] = noise[i - 1] + step + restoring
        noise[i] = np.clip(noise[i], 0, 1)

    return noise

def first_crossing(series, times, threshold):
    idx = np.where(series >= threshold)[0]
    if len(idx) == 0:
        return np.nan
    return float(times[idx[0]])

## 4. Single-trial comparison

In [ ]:
t = np.linspace(0, 10, 201)
d0 = 0.02
threshold = 0.5

linear = linear_baseline(t, d0=d0)
nonlinear = nonlinear_amplification(t, d0=d0, k=0.65)
noise = stochastic_noise(t, d0=d0, sigma=0.006, pullback=0.06, seed=260614806)

single_trial = pd.DataFrame({
    "time": t,
    "linear_baseline": linear,
    "deterministic_nonlinear": nonlinear,
    "stochastic_noise": noise
})

single_trial.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(single_trial["time"], single_trial["linear_baseline"], label="Linear baseline")
ax.plot(single_trial["time"], single_trial["deterministic_nonlinear"], label="Deterministic nonlinear amplification")
ax.plot(single_trial["time"], single_trial["stochastic_noise"], label="Stochastic noise comparison", alpha=0.8)

ax.axhline(threshold, linestyle="--", linewidth=1.5, label="Decision threshold")

ax.set_xlabel("Toy time")
ax.set_ylabel("Distinguishability")
ax.set_title("Noise fluctuations vs reliable nonlinear amplification")
ax.set_ylim(0, 1.05)
ax.legend()

single_figure_path = FIGURES / "37_noise_vs_nonlinearity.png"
fig.savefig(single_figure_path, dpi=200, bbox_inches="tight")
plt.show()

single_figure_path

## 5. Multi-trial reliability

A computational mechanism needs reliability, not a lucky fluctuation.

We run many noisy trials and compare their threshold-crossing behavior against the deterministic nonlinear trajectory.

In [ ]:
n_trials = 200
sigma = 0.006
pullback = 0.06

records = []

for trial in range(n_trials):
    trajectory = stochastic_noise(
        t,
        d0=d0,
        sigma=sigma,
        pullback=pullback,
        seed=trial
    )

    records.append({
        "trial": trial,
        "model": "stochastic_noise",
        "first_crossing_time": first_crossing(trajectory, t, threshold),
        "max_distinguishability": float(np.max(trajectory))
    })

records.append({
    "trial": -1,
    "model": "linear_baseline",
    "first_crossing_time": first_crossing(linear, t, threshold),
    "max_distinguishability": float(np.max(linear))
})

records.append({
    "trial": -1,
    "model": "deterministic_nonlinear",
    "first_crossing_time": first_crossing(nonlinear, t, threshold),
    "max_distinguishability": float(np.max(nonlinear))
})

reliability = pd.DataFrame(records)
reliability.head()

In [ ]:
summary = reliability.groupby("model").agg(
    trials=("trial", "count"),
    crossing_rate=("first_crossing_time", lambda s: float(np.mean(~np.isnan(s)))),
    median_crossing_time=("first_crossing_time", "median"),
    max_distinguishability_mean=("max_distinguishability", "mean"),
    max_distinguishability_max=("max_distinguishability", "max")
).reset_index()

summary

## 6. Reliability figure

This plot shows maximum distinguishability across noisy trials, with deterministic baselines overlaid.

The important contrast is not whether noise ever moves. It moves.

The question is whether the process reliably reaches the decision threshold.

In [ ]:
noise_only = reliability[reliability["model"] == "stochastic_noise"].copy()

fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(
    noise_only["trial"],
    noise_only["max_distinguishability"],
    alpha=0.65,
    label="Stochastic noise trials"
)

ax.axhline(threshold, linestyle="--", linewidth=1.5, label="Decision threshold")
ax.axhline(float(np.max(linear)), linewidth=1.5, label="Linear baseline max")
ax.axhline(float(np.max(nonlinear)), linewidth=1.5, label="Deterministic nonlinear max")

ax.set_xlabel("Noise trial")
ax.set_ylabel("Maximum distinguishability")
ax.set_title("Threshold reliability: noise fluctuations vs nonlinear amplification")
ax.set_ylim(0, 1.05)
ax.legend()

reliability_figure_path = FIGURES / "37_threshold_reliability.png"
fig.savefig(reliability_figure_path, dpi=200, bbox_inches="tight")
plt.show()

reliability_figure_path

## 7. Interpretation

The toy model illustrates a distinction:

- stochastic noise can fluctuate,
- deterministic amplification can reliably separate nearby states,
- reliable separation supports decision extraction,
- decision extraction is what matters computationally.

In [ ]:
model_comparison = pd.DataFrame([
    {
        "model": "Linear baseline",
        "behavior": "Distinguishability remains near its initial value.",
        "decision_relevance": "No reliable threshold crossing."
    },
    {
        "model": "Deterministic nonlinear amplification",
        "behavior": "Distinguishability grows systematically.",
        "decision_relevance": "Reliable threshold crossing supports decision extraction."
    },
    {
        "model": "Stochastic noise comparison",
        "behavior": "Distinguishability fluctuates.",
        "decision_relevance": "Fluctuations alone do not provide reliable computational amplification."
    }
])

model_comparison

## 8. Engineering statement

In [ ]:
engineering_statement = {
    "statement": "Reliable decision extraction requires repeatable threshold crossing.",
    "observation": "Noise can fluctuate without producing systematic amplification.",
    "context": "Nonlinear state-dependent dynamics may amplify distinguishability reliably.",
    "next_step": "Compare toy nonlinear amplification with physically motivated stochastic models."
}

pd.DataFrame([engineering_statement]).T.rename(columns={0: "text"})

## 9. Bottom line

Noise is not the same thing as computationally useful amplification.

A stochastic process may occasionally increase distinguishability, but reliable computation requires repeatable threshold crossing that can support decision extraction.

In [ ]:
bottom_line = (
    "Noise is not the same thing as computationally useful amplification. "
    "A stochastic process may occasionally increase distinguishability, but reliable computation "
    "requires repeatable threshold crossing that can support decision extraction."
)

bottom_line

## 10. Export artifacts

This notebook exports:

- `results/37_single_trial.csv`
- `results/37_reliability_trials.csv`
- `results/37_reliability_summary.csv`
- `results/37_model_comparison.csv`
- `results/37_noise_vs_nonlinearity.md`
- `figures/37_noise_vs_nonlinearity.png`
- `figures/37_threshold_reliability.png`

In [ ]:
single_csv = RESULTS / "37_single_trial.csv"
reliability_csv = RESULTS / "37_reliability_trials.csv"
summary_csv = RESULTS / "37_reliability_summary.csv"
comparison_csv = RESULTS / "37_model_comparison.csv"
md_path = RESULTS / "37_noise_vs_nonlinearity.md"

single_trial.to_csv(single_csv, index=False)
reliability.to_csv(reliability_csv, index=False)
summary.to_csv(summary_csv, index=False)
model_comparison.to_csv(comparison_csv, index=False)

report = []
report.append("# 37 Noise vs Nonlinearity")
report.append("")
report.append("## Purpose")
report.append(
    "This notebook compares deterministic state-dependent amplification against stochastic fluctuations."
)
report.append("")
report.append("## Reliability summary")

for _, row in summary.iterrows():
    report.append(
        f"- **{row['model']}** — crossing rate: {row['crossing_rate']}; "
        f"median crossing time: {row['median_crossing_time']}; "
        f"mean max distinguishability: {row['max_distinguishability_mean']}"
    )

report.append("")
report.append("## Model comparison")

for _, row in model_comparison.iterrows():
    report.append(
        f"- **{row['model']}** — {row['behavior']} {row['decision_relevance']}"
    )

report.append("")
report.append("## Engineering statement")

for key, value in engineering_statement.items():
    report.append(f"- **{key.replace('_', ' ').title()}**: {value}")

report.append("")
report.append("## Bottom line")
report.append(bottom_line)

md_path.write_text("\n".join(report), encoding="utf-8")

print(f"Saved {single_csv}")
print(f"Saved {reliability_csv}")
print(f"Saved {summary_csv}")
print(f"Saved {comparison_csv}")
print(f"Saved {md_path}")
print(f"Saved {single_figure_path}")
print(f"Saved {reliability_figure_path}")

## 11. Optional downloads

In [ ]:
downloads = [
    single_csv,
    reliability_csv,
    summary_csv,
    comparison_csv,
    md_path,
    single_figure_path,
    reliability_figure_path,
]

try:
    from google.colab import files
    for path in downloads:
        files.download(str(path))
except Exception:
    print("Saved artifacts:")
    for path in downloads:
        print(path)